In [49]:
import pandas as pd

In [50]:
employee = {'id':[1,2,3]
           ,'salary':[100,200,300]}
employee = pd.DataFrame(employee)
employee

,id,salary
0,1,100
1,2,200
2,3,300


In [51]:
df_sorted.values

array([300, 200, 100])

In [52]:
top2_values = employee['salary'].drop_duplicates().nlargest(2).reset_index(drop=True)
top2_value = top2_values[1] if len(top2_values) else None
outcome= pd.DataFrame({'SecondHighestSalary':[top2_value]})

In [53]:
outcome

,SecondHighestSalary
0,200


In [60]:
#alternative way. use drop duplicates
person = pd.DataFrame({'id':[1,2,3]
                      ,'email':['john@example.com','bob@example.com','john@example.com']})

def drop_duplicate_emails(person: pd.DataFrame) -> None:
    person.sort_values(by=['id'],ascending=True,inplace=True)
    person.drop_duplicates(subset='email',inplace=True)
    
drop_duplicate_emails(person)    
person

,id,email
0,1,john@example.com
1,2,bob@example.com


### Use of melt

In [70]:
products = pd.DataFrame({'product_id':[0,1]
                        ,'product_name':['product_a','product_b']
                        ,'store1':[95,70]
                        ,'store2':[100,None]
                        ,'store3':[105,80]})
products

,product_id,product_name,store1,store2,store3
0,0,product_a,95,100.0,105
1,1,product_b,70,NaN,80


In [71]:

products_melted = products.melt(id_vars=['product_id','product_name'], #
                                value_vars=['store1','store2','store3'], #value_vars
                                var_name='store', #
                                value_name='price'
                                )
products_melted = products_melted[~products_melted.price.isna()].reset_index(drop=True)
products_melted

,product_id,product_name,store,price
0,0,product_a,store1,95.0
1,1,product_b,store1,70.0
2,0,product_a,store2,100.0
3,0,product_a,store3,105.0
4,1,product_b,store3,80.0


In [79]:
#go in the other direction:
products_melted.pivot(index=['product_id','product_name'],columns='store',values='price').reset_index()

store,product_id,product_name,store1,store2,store3
0,0,product_a,95.0,100.0,105.0
1,1,product_b,70.0,NaN,80.0


In [81]:
products_melted.product_id.unique()

array([0, 1])

## Adds

In [89]:
import pandas as pd

Ads table:
+-------+---------+---------+
| ad_id | user_id | action  |
+-------+---------+---------+
| 1     | 1       | Clicked |
| 2     | 2       | Clicked |
| 3     | 3       | Viewed  |
| 5     | 5       | Ignored |
| 1     | 7       | Ignored |
| 2     | 7       | Viewed  |
| 3     | 5       | Clicked |
| 1     | 4       | Viewed  |
| 2     | 11      | Viewed  |
| 1     | 2       | Clicked |

+-------+---------+---------+

In [90]:
ads = pd.DataFrame({'ad_id':[1,2,3,5,1,2,3,1,2,1]
      ,'user_id':[1,2,3,5,7,7,5,4,11,2]
      ,'action':['Clicked','Clicked','Viewed','Ignored','Ignored','Viewed','Clicked','Viewed','Viewed','Clicked']})

ads_grouped = ads.groupby(['ad_id','action']).size().to_frame('action_count').reset_index()
#use pivot
ads_grouped = ads_grouped.pivot(index='ad_id',columns='action',values='action_count')
ads_grouped

action,Clicked,Ignored,Viewed
ad_id,,,
1,2.0,1.0,1.0
2,1.0,NaN,2.0
3,1.0,NaN,1.0
5,NaN,1.0,NaN


In [91]:
#assumes clicked and viewed exist for both!
ads_grouped = ads.groupby(['ad_id','action']).size().to_frame('action_count').reset_index()
#use pivot
ads_grouped = ads_grouped.pivot(index='ad_id',columns='action',values='action_count').reset_index()
ads_grouped = ads_grouped.fillna(0)
ads_grouped['denominator'] = ads_grouped.Clicked + ads_grouped.Viewed
ads_grouped['ctr'] = ads_grouped.apply(lambda x: round(x['Clicked']/x['denominator']*100,2) if x['denominator']>0 else 0,axis=1)
ads_grouped = ads_grouped.sort_values(by=['ctr','ad_id'],ascending=[False,True])
ads_grouped[['ad_id','ctr']]

action,ad_id,ctr
0,1,66.67
2,3,50.00
1,2,33.33
3,5,0.00


In [97]:
ads = pd.DataFrame({'ad_id':[1,2,3,5,1,2,3,1,2,1]
      ,'user_id':[1,2,3,5,7,7,5,4,11,2]
      ,'action':['Clicked','Clicked','Viewed','Ignored','Ignored','Viewed','Clicked','Viewed','Viewed','Clicked']})


In [103]:
#remove ignore
ads_flt = ads[~(ads.action == 'Ignored')].reset_index(drop=True)

#get only those that clicked
ads_clicked = ads_flt[ads_flt.action == 'Clicked'].reset_index(drop=True)
ads_clicked = ads_clicked.groupby('ad_id').size().to_frame('Clicked').reset_index()

#get clicked + viewed
ads_sum = ads_flt.groupby('ad_id').size().to_frame('denominator').reset_index()

#create df, use initial dataset with ignore to include 0s
ads_ctr = ads[['ad_id']].drop_duplicates().reset_index(drop=True)
ads_ctr = ads_ctr.merge(ads_sum,how='left',on='ad_id').merge(ads_clicked,how='left',on='ad_id').fillna(0)
ads_ctr['ctr'] = ads_ctr.apply(lambda x: round(100*x['Clicked']/x['denominator'],2) if x['denominator']>0 else 0,axis=1)
ads_ctr[['ad_id','ctr']].sort_values(by=['ctr','ad_id'],ascending=[False,True])

,ad_id,ctr
0,1,66.67
2,3,50.00
1,2,33.33
3,5,0.00


In [113]:
ads_ctr.groupby('ad_id')['ctr'].agg('sum')

ad_id
1    66.67
2    33.33
3    50.00
5     0.00
Name: ctr, dtype: float64